#### **1. 總體架構演進**

為實現從「每日宏觀分析」到「高頻微觀監控」的戰略目標，我們將目前的單體應用 (`stress_report_app`) 進行職責分離與功能擴充。後端系統將演變為一個由多個獨立微應用組成的生態系統，每個應用專注於一項核心任務。

本次升級將新增兩個核心的**數據擷取微應用**：

1.  **`apps/hf_data_ingestor`**: 專門負責從 `yfinance` 獲取、快取並儲存高頻市場數據。
2.  **`apps/file_processor`**: 專門負責解析、清洗並儲存由使用者手動上傳的交易所數據檔案。

現有的 `apps/stress_report_app` 將不再直接对外請求數據，而是轉為從這兩個新應用所建立的本地資料庫中讀取數據，專注於分析與報告生成。

In [ ]:
# -*- coding: utf-8 -*-
# @title 🐺 全景市場分析儀 - v10.1 (SOP合規調度版)
# @markdown ### ▶️ 步驟一：掛載 Google Drive
# @markdown **點擊執行，並在跳出的視窗中授權存取您的 Google Drive。**
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ==============================================================================
# @markdown ---
# @markdown ### ▶️ 步驟二：設定分析模式與參數
# @markdown **請先選擇分析模式，然後設定相應參數，最後點擊執行按鈕。**
# ==============================================================================

# --- 核心依賴安裝 ---
!pip install -q tqdm pytz yfinance duckdb pandas # 新增 yfinance, duckdb, pandas 到 Colab 環境

import os
import sys
import subprocess
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Markdown
from datetime import datetime, timedelta
import pytz
import time
from tqdm.notebook import tqdm
import glob

# --- 全局變數與路徑設定 ---
TAIPEI_TZ = pytz.timezone('Asia/Taipei')
PROJECT_PATH = "/content/wolf_date_analyzer" # 後端 Python 專案的本地複製路徑
GDRIVE_ROOT = "/content/drive/MyDrive/Colab_Wolf_Analyzer" # Google Drive 中的專案根目錄
PERSISTENT_DATA_PATH = os.path.join(GDRIVE_ROOT, "data_workspace") # 持久化數據目錄 (例如資料庫、快取)
UPLOADS_PATH = os.path.join(GDRIVE_ROOT, "uploads") # 使用者上傳檔案的目錄 (file_processor 會掃描此處)

# --- 確保所有雲端目錄存在 ---
os.makedirs(PERSISTENT_DATA_PATH, exist_ok=True)
os.makedirs(os.path.join(PERSISTENT_DATA_PATH, "output", "reports"), exist_ok=True)
# os.makedirs(os.path.join(PERSISTENT_DATA_PATH, "raw_data"), exist_ok=True) # 舊的，可能不再需要
os.makedirs(os.path.join(PERSISTENT_DATA_PATH, "logs"), exist_ok=True)
os.makedirs(os.path.join(PERSISTENT_DATA_PATH, "cache", "yfinance"), exist_ok=True) # hf_data_ingestor 的快取目錄
os.makedirs(UPLOADS_PATH, exist_ok=True) # 確保上傳目錄存在

GITHUB_REPO_URL = "https://github.com/hsp1234-web/0629_WOLF_DATE.git" #@param {type:"string"} # 您的後端程式碼庫 URL
TARGET_BRANCH = "feature/jules-sop4-verification-繁體中文" #@param {type:"string"} # 您的後端程式碼庫分支

execution_params = {} # 用於儲存使用者選擇的執行參數

# --- (日誌與子程序執行函數) ---
def log_message(message, level="INFO"):
    timestamp = datetime.now(TAIPEI_TZ).strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{timestamp} Asia/Taipei] [{level}] {message}")

def run_and_log_subprocess(cmd, env=None, title=""):
    log_message(f"--- 開始執行: {title} ---")
    log_message(f"🔩 執行指令: {' '.join(cmd)}")
    start_time = time.time()
    process = subprocess.Popen(
        cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1
    )
    pbar = tqdm(total=None, desc=f"⏳ {title}", bar_format="{l_bar}{bar}| {elapsed}")
    for line in iter(process.stdout.readline, ''):
        log_message(line.strip()) # 打印子程序的即時輸出
        pbar.update(0) # 更新進度條動畫
    process.wait()
    pbar.close()
    end_time = time.time()
    elapsed = end_time - start_time
    log_message(f"--- {title} 完成 ---")
    log_message(f"✅ 返回碼: {process.returncode} | 耗時: {elapsed:.2f} 秒")
    print("\n" + "="*80 + "\n")
    return process.returncode

# --- UI 介面生成 ---
def create_interactive_ui():
    """創建並顯示全新的、動態的互動介面"""
    global execution_params
    style = {'description_width': 'initial'}
    layout = widgets.Layout(width='auto', margin='5px')

    # --- 模式選擇 ---
    mode_selector = widgets.RadioButtons(
        options=[('每日宏觀趨勢報告 (舊)', 'macro_report'),
                 ('高頻數據擷取 (yfinance)', 'hf_ingest'),
                 ('處理上傳的交易所檔案', 'file_process')],
        description='選擇執行任務:',
        style=style
    )

    # --- 各模式的參數 ---
    # A. 宏觀報告參數 (舊 stress_report_app)
    macro_start_date = widgets.DatePicker(description='開始日期:', value=datetime.now().date() - timedelta(days=90), style=style, layout=layout)
    macro_end_date = widgets.DatePicker(description='結束日期:', value=datetime.now().date(), style=style, layout=layout)
    macro_db_path_input = widgets.Text(value=os.path.join(PERSISTENT_DATA_PATH, "market_data.duckdb"), description='數據庫路徑:', style=style, layout=layout, disabled=True)
    macro_params = widgets.VBox([
        widgets.HTML("<h4>設定宏觀報告參數 (舊應用)：</h4>"),
        widgets.HBox([macro_start_date, macro_end_date]),
        macro_db_path_input,
        widgets.HTML("<i><small>此報告將從上述數據庫讀取由 '高頻數據擷取' 或 '檔案處理' 應用準備的數據。</small></i>")
    ])

    # B. 高頻擷取參數 (hf_data_ingestor)
    hf_interval = widgets.Dropdown(options=['1m', '5m', '15m', '30m', '1h', '1d'], value='5m', description='數據顆粒度:', style=style, layout=layout)
    hf_tickers = widgets.Text(value='^VIX,SPY,TLT,BTC-USD', description='關注標的 (逗號分隔):', style=style, layout=layout)
    hf_db_path_input = widgets.Text(value=os.path.join(PERSISTENT_DATA_PATH, "market_data.duckdb"), description='目標資料庫路徑:', style=style, layout=layout)
    hf_params = widgets.VBox([
        widgets.HTML("<h4>設定高頻數據擷取參數 (yfinance)：</h4>"),
        hf_interval,
        hf_tickers,
        hf_db_path_input
    ])

    # C. 檔案處理參數 (file_processor)
    fp_uploads_path_label = widgets.HTML(f"<p><b>掃描處理目錄:</b> <code>{UPLOADS_PATH}</code></p><p><i><small>請先將交易所數據檔案手動上傳至此 Google Drive 目錄。</small></i></p>")
    fp_db_path_input = widgets.Text(value=os.path.join(PERSISTENT_DATA_PATH, "market_data.duckdb"), description='目標資料庫路徑:', style=style, layout=layout)
    fp_params = widgets.VBox([
        widgets.HTML("<h4>設定交易所檔案處理任務：</h4>"),
        fp_uploads_path_label,
        fp_db_path_input
    ])

    # --- 動態顯示邏輯 ---
    param_widgets_container = widgets.VBox([macro_params]) # 預設顯示宏觀報告參數
    def on_mode_change(change):
        mode = change.new
        if mode == 'macro_report':
            param_widgets_container.children = [macro_params]
        elif mode == 'hf_ingest':
            param_widgets_container.children = [hf_params]
        elif mode == 'file_process':
            param_widgets_container.children = [fp_params]
    mode_selector.observe(on_mode_change, names='value')

    # --- 通用設定與執行按鈕 ---
    run_button = widgets.Button(description="🚀 開始執行任務", button_style='success', icon='play', layout=widgets.Layout(width='200px'))
    output_area = widgets.Output() # 用於顯示子程序輸出的區域

    def on_run_button_clicked(b):
        with output_area:
            clear_output(wait=True) # 清除上一次的輸出
            log_message("參數已確認，準備執行...")
            execution_params['mode'] = mode_selector.value

            # 根據模式收集參數
            if mode_selector.value == 'macro_report':
                execution_params['start_date'] = macro_start_date.value.strftime('%Y-%m-%d')
                execution_params['end_date'] = macro_end_date.value.strftime('%Y-%m-%d')
                execution_params['db_path'] = macro_db_path_input.value # stress_report_app 也需要知道資料庫路徑
            elif mode_selector.value == 'hf_ingest':
                execution_params['interval'] = hf_interval.value
                execution_params['tickers'] = hf_tickers.value
                execution_params['db_path'] = hf_db_path_input.value
            elif mode_selector.value == 'file_process':
                execution_params['input_dir'] = UPLOADS_PATH # 固定為雲端上傳目錄
                execution_params['db_path'] = fp_db_path_input.value
            
            run_main_execution() # 呼叫主執行函數

    run_button.on_click(on_run_button_clicked)

    # 顯示所有 UI 元件
    display(widgets.VBox([
        widgets.HTML("<h2>步驟一：選擇執行任務</h2>"),
        mode_selector, 
        widgets.HTML("<hr>"),
        widgets.HTML("<h2>步驟二：設定任務參數</h2>"),
        param_widgets_container, 
        widgets.HTML("<hr>"),
        widgets.HTML("<h2>步驟三：執行</h2>"),
        run_button, 
        output_area # 將 output_area 放在這裡以顯示日誌
    ]))

# --- 主執行流程 ---
def run_main_execution():
    display(HTML('<h3 style="color:#82B1FF;">⏳ 環境準備與程式碼同步中...</h3>'))

    # 1. Git 和環境準備 (如果後端程式碼有更新，則重新拉取)
    if os.path.exists(PROJECT_PATH):
        log_message(f"專案目錄 {PROJECT_PATH} 已存在，將執行 git pull 更新...")
        os.chdir(PROJECT_PATH)
        pull_cmd = ["git", "pull", "origin", TARGET_BRANCH]
        if run_and_log_subprocess(pull_cmd, title="Git 專案更新") != 0:
            log_message("Git 更新失敗，嘗試刪除後重新 clone...", level="WARNING")
            os.chdir('/content/') # 返回上層目錄以便刪除
            shutil.rmtree(PROJECT_PATH)
            clone_repo = True
        else:
            clone_repo = False # 更新成功，無需重新 clone
    else:
        clone_repo = True
    
    if clone_repo:
        os.chdir('/content/') # 確保在正確的目錄下 clone
        git_cmd = ["git", "clone", "--branch", TARGET_BRANCH, "--single-branch", GITHUB_REPO_URL, PROJECT_PATH]
        if run_and_log_subprocess(git_cmd, title="Git 專案拉取") != 0: 
            log_message("Git 專案拉取失敗！請檢查網路連線、倉儲 URL 和分支名稱。", level="ERROR")
            return
    
    os.chdir(PROJECT_PATH) # 切換到專案目錄
    log_message(f"目前工作目錄: {os.getcwd()}")

    # 2. 建立從 Google Drive 到專案內部 data_workspace 的符號連結 (如果不存在)
    # 這使得 Python 腳本可以像存取本地路徑一樣存取 Drive 中的數據
    local_data_path = os.path.join(PROJECT_PATH, "data_workspace")
    if not os.path.exists(local_data_path):
        try:
            os.symlink(PERSISTENT_DATA_PATH, local_data_path, target_is_directory=True)
            log_message(f"成功建立從 {PERSISTENT_DATA_PATH} 到 {local_data_path} 的符號連結。")
        except Exception as e:
            log_message(f"建立符號連結失敗: {e}。請確認 PERSISTENT_DATA_PATH ({PERSISTENT_DATA_PATH}) 存在。", level="ERROR")
            # 某些環境可能不支援 symlink，或者 Drive 檔案系統的限制
            # 作為備案，可以考慮直接使用 PERSISTENT_DATA_PATH，但這需要修改後端腳本的路徑邏輯
            # 或者，將 PERSISTENT_DATA_PATH 的內容複製到 local_data_path (較慢，且可能佔用 Colab 空間)
            # display(HTML('<p style="color:red;">錯誤：無法建立 data_workspace 符號連結。部分功能可能無法正常運作。</p>'))
            # return # 嚴重錯誤，終止執行
            log_message("警告: 符號連結建立失敗，後端腳本可能需要直接使用 Drive 路徑。", level="WARNING")
            # 即使連結失敗，我們仍然嘗試執行，假設後端能處理絕對路徑

    # 3. 安裝專案依賴 (requirements.txt)
    # 決定使用哪個 requirements.txt，或是合併它們
    # 為簡單起見，這裡假設有一個根目錄的 requirements.txt 包含所有通用依賴
    # 或者，根據執行的 app 動態選擇其 requirements.txt
    # 目前的後端結構，每個 app 有自己的 requirements.txt
    # 我們需要根據選擇的模式來安裝對應的依賴
    requirements_file_path = ""
    mode = execution_params['mode']
    if mode == 'macro_report':
        # 假設 stress_report_app 也有一個 requirements.txt
        # 如果 stress_report_app 依賴於 hf_data_ingestor 和 file_processor 產生的數據庫，
        # 它自身可能只需要 duckdb 和 pandas 等分析庫
        requirements_file_path = os.path.join("apps", "stress_report_app", "requirements.txt") 
        # 為了確保 stress_report_app 能讀取 duckdb，也安裝其依賴
        # !pip install -q duckdb pandas matplotlib # 假設
    elif mode == 'hf_ingest':
        requirements_file_path = os.path.join("apps", "hf_data_ingestor", "requirements.txt")
    elif mode == 'file_process':
        requirements_file_path = os.path.join("apps", "file_processor", "requirements.txt")
    
    if requirements_file_path and os.path.exists(requirements_file_path):
        log_message(f"準備安裝依賴: {requirements_file_path}")
        # 注意：Colab 環境中 pip install 通常是全域的。
        # 多次執行不同 app 的依賴安裝是安全的。
        pip_cmd = [sys.executable, "-m", "pip", "install", "-r", requirements_file_path]
        if run_and_log_subprocess(pip_cmd, title=f"安裝 {mode} 依賴") != 0: 
            log_message(f"安裝 {mode} 依賴失敗！", level="ERROR")
            return
    elif requirements_file_path: # 路徑指定了但檔案不存在
        log_message(f"警告: 依賴檔案 {requirements_file_path} 未找到。跳過此應用的特定依賴安裝。", level="WARNING")
    else: # requirements_file_path 為空 (例如 stress_report_app 沒有獨立的 requirements.txt)
        log_message(f"模式 {mode} 沒有指定特定的 requirements.txt，假設通用依賴已滿足。", level="INFO")

    display(HTML('<h3 style="color:#82B1FF;">✅ 環境準備就緒，開始執行後端腳本...</h3>'))

    # --- 全新的、根據模式產生指令的邏輯 ---
    cmd = [sys.executable, "-m"] # 使用 python -m 來執行模組
    title = ""
    db_path_for_app = execution_params.get('db_path', os.path.join(PERSISTENT_DATA_PATH, "market_data.duckdb"))

    if mode == 'macro_report':
        # 假設 stress_report_app 的入口是 apps.stress_report_app.run
        # 並且它現在從指定的 db_path 讀取數據
        cmd.extend(["apps.stress_report_app.run"]) 
        cmd.extend(["--start-date", execution_params['start_date']])
        cmd.extend(["--end-date", execution_params['end_date']])
        cmd.extend(["--db-path", db_path_for_app]) # 傳遞資料庫路徑給舊應用
        # 假設報告輸出到 PERSISTENT_DATA_PATH 下的 output/reports
        cmd.extend(["--output-dir", os.path.join(PERSISTENT_DATA_PATH, "output", "reports")])
        title = "執行宏觀趨勢報告 (舊應用)"
    elif mode == 'hf_ingest':
        cmd.extend(["apps.hf_data_ingestor.run"])
        # 確保 tickers 字串被正確傳遞，如果包含特殊字元或空格，可能需要額外處理
        # subprocess 通常會處理好空格分隔的參數，但如果 tickers 本身包含空格且用逗號分隔，
        # yfinance 可能無法正確解析。建議 tickers 不要包含內部空格，或用引號包住單個 ticker。
        # argparse 通常能處理好 "^VIX,SPY,TLT" 這樣的字串。
        cmd.extend(["--tickers", execution_params['tickers']])
        cmd.extend(["--interval", execution_params['interval']])
        cmd.extend(["--db-path", db_path_for_app])
        title = "執行高頻數據擷取 (yfinance)"
    elif mode == 'file_process':
        cmd.extend(["apps.file_processor.run"])
        cmd.extend(["--input-dir", execution_params['input_dir']]) # UPLOADS_PATH
        cmd.extend(["--db-path", db_path_for_app])
        title = "執行交易所檔案處理"
    else:
        log_message(f"錯誤：未知的執行模式 '{mode}'", level="ERROR")
        return

    # 執行選定的後端腳本
    return_code = run_and_log_subprocess(cmd, title=title)

    # --- 最終結果判斷 ---
    if return_code == 0:
        log_message(f"🎉 任務 '{title}' 成功完成！腳本以返回碼 0 正常結束。")
        # 未來可以加入報告預覽邏輯，例如：
        if mode == 'macro_report':
            # 假設報告生成在 PERSISTENT_DATA_PATH/output/reports/some_report.html
            # 列出可能的報告檔案
            report_dir = os.path.join(PERSISTENT_DATA_PATH, "output", "reports")
            # 簡單查找最新的 html 檔案作為範例
            list_of_files = glob.glob(os.path.join(report_dir, '*.html')) 
            if list_of_files:
                latest_file = max(list_of_files, key=os.path.getctime)
                display(HTML(f'<p>報告已生成，可在此處查看 (請確認路徑和檔案名稱)：<a href="{latest_file.replace(GDRIVE_ROOT, "/content/drive/MyDrive/" + os.path.basename(GDRIVE_ROOT))}" target="_blank">{os.path.basename(latest_file)}</a></p>'))
                # 注意：直接從 /content/drive/... 的連結可能無法在 Colab 中直接點擊預覽，
                # 使用者可能需要手動到 Google Drive 找到檔案。
                # 也可以考慮將報告內容讀取並直接顯示在 Colab Output 中 (如果報告是 HTML 且適合內嵌)。
            else:
                 log_message("宏觀報告模式執行完畢，但在預期目錄未找到 HTML 報告檔案。", level="INFO")
    else:
        log_message(f"🔥 任務 '{title}' 執行失敗！腳本以返回碼 {return_code} 異常結束。請檢查上方日誌輸出以了解詳細錯誤訊息。", level="ERROR")

    log_message("--- Colab 端全部流程結束 ---")

# --- 主函數入口 ---
if __name__ == '__main__':
    # 確保 shutil 被匯入 (如果之前沒有，run_main_execution 中用到了)
    import shutil 
    create_interactive_ui() # 啟動互動介面
else:
    # 如果不是作為主腳本執行 (例如被 Colab 匯入)，也嘗試啟動 UI
    # 這有助於在某些 Colab 環境下直接運行儲存格時也能顯示介面
    import shutil 
    create_interactive_ui()